# Perseus MCP Tool Handbook: Tutorial and Complete Reference

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Purpose and place in the tutorial series</a>
* <a href="#catalog-concepts">2 - How an MCP tool catalog works</a>
* <a href="#setup">3 - Install dependencies and load the server</a>
* <a href="#helpers">4 - Result and schema helper functions</a>
* <a href="#connect">5 - Connect and retrieve the live tool catalog</a>
* <a href="#categories">6 - Understand the six tool families</a>
* <a href="#schema-reading">7 - Learn to read a JSON input schema</a>
* <a href="#quick-reference">8 - Complete quick-reference table</a>
* <a href="#decision-guide">9 - Choose a tool from a research question</a>
* <a href="#examples">10 - Example argument templates for every tool</a>
* <a href="#local-call">11 - Tutorial call: inspect local cache status</a>
* <a href="#live-call">12 - Tutorial call: discover an author</a>
* <a href="#result-types">13 - Parse JSON, XML, plaintext, and upstream responses</a>
* <a href="#validation">14 - Validate arguments before calling</a>
* <a href="#catalog-query">15 - Search and filter the catalog programmatically</a>
* <a href="#drift-check">16 - Detect documentation and server drift</a>
* <a href="#detailed-reference">17 - Detailed generated reference for all tools</a>
* <a href="#machine-readable">18 - Build a machine-readable reference catalog</a>
* <a href="#operations">19 - Network, cache, cost, and safety considerations</a>
* <a href="#external-clients">20 - Use the same catalog from external MCP clients</a>
* <a href="#troubleshooting">21 - Troubleshooting</a>
* <a href="#next-steps">22 - Continue learning</a>
* <a href="#sources">23 - Sources</a>
* <a href="#required-libraries">24 - Required libraries</a>
* <a href="#notebook-version">25 - Notebook version</a>

## 1 - Purpose and place in the tutorial series <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook is both a **tutorial** and a **reference handbook** for the complete `perseus` MCP tool surface.

The preceding notebooks teach workflows:

- notebook `01_` explains plain Perseus CTS;
- notebook `02_` explains plain CTS/Scaife search concepts;
- notebook `03_` introduces an MCP client and a discovery-to-passage workflow;
- notebook `04_` teaches MCP Greek search and CTS navigation.

This notebook changes perspective. Instead of following one research question, it teaches how to understand and inspect the **entire live tool catalog**. It answers questions such as:

- What tools are currently registered?
- Which tool should I choose for a particular research task?
- Which arguments are required, optional, nullable, or defaulted?
- Does a tool return JSON, XML, or plaintext?
- Does it use Perseus CTS, Scaife, or only the local cache?
- Which calls are lightweight, large, cached, or state-changing?
- How can code detect when the server and this reference drift apart?

Most of the reference is generated from `client.list_tools()`, so names, descriptions, and JSON schemas reflect the server loaded in the current kernel rather than a copied static list.

## 2 - How an MCP tool catalog works <a class="anchor" id="catalog-concepts"></a>
##### [Back to ToC](#TOC)

An MCP server advertises capabilities as **tools**. Each tool entry normally contains:

| Field | Purpose |
|---|---|
| `name` | Stable identifier used by `client.call_tool(...)` |
| `description` | Natural-language explanation for people and tool-using models |
| `inputSchema` | JSON Schema describing accepted arguments |
| Additional metadata | Optional annotations supplied by the MCP implementation |

A typical interaction is:

```python
async with Client(mcp) as client:
    tools = await client.list_tools()
    result = await client.call_tool(
        "get_passage_plaintext",
        {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"},
    )
```

The catalog is not merely documentation. A client can use it to construct interfaces, validate calls, expose tools to an LLM, generate forms, or build automated reference pages.

Every tool in this project returns content as text blocks. The text may itself contain JSON, XML, HTML from an imperfect upstream response, or readable plaintext. The caller must know which representation to expect.

## 3 - Install dependencies and load the server <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

The first cell installs Perseus MCP into the active Jupyter kernel. It defaults to the local repository in editable mode for development-branch work; change the switch to `"pypi"` for the published package. The setup then locates the repository root, configures the shared metadata cache, imports and reloads `perseus_mcp.server`, and obtains its FastMCP object.

Set `PERSEUS_MCP_INSTALL_SOURCE` in the install cell to `"repo"` for editable development-branch work or `"pypi"` for the published package.

The notebook uses an in-process client exactly like notebooks `03_` and `04_`. Importing the module does not launch a separate command-line server.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Use "repo" while working on this development checkout.
# Use "pypi" to run against the published package normal users install.
PERSEUS_MCP_INSTALL_SOURCE = "repo"  # "repo" or "pypi"
PERSEUS_MCP_PYPI_SPEC = "perseus-mcp"

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

install_source = PERSEUS_MCP_INSTALL_SOURCE.lower()
if install_source == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find the Perseus-mcp repository from {START}. Open this notebook inside the repository checkout or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    package_target = ["--editable", str(REPO_ROOT)]
    package_label = f"editable repository at {REPO_ROOT}"
elif install_source == "pypi":
    package_target = ["--force-reinstall", PERSEUS_MCP_PYPI_SPEC]
    package_label = PERSEUS_MCP_PYPI_SPEC
else:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *package_target,
        "python-dotenv>=1.0.0",
    ]
)

print(f"Installed perseus-mcp from {package_label} into this kernel")

In [ ]:
from collections import Counter
from pathlib import Path
import importlib
import json
import os
import sys

from IPython.display import Markdown, display

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

PERSEUS_MCP_INSTALL_SOURCE = globals().get("PERSEUS_MCP_INSTALL_SOURCE", "repo").lower()
if PERSEUS_MCP_INSTALL_SOURCE not in {"repo", "pypi"}:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

if PERSEUS_MCP_INSTALL_SOURCE == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    SRC_DIR = REPO_ROOT / "src"
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
elif REPO_ROOT is not None:
    SRC_DIR = REPO_ROOT / "src"
    src_dir_resolved = SRC_DIR.resolve()

    def _is_repo_src(path_entry):
        try:
            return Path(path_entry).resolve() == src_dir_resolved
        except (OSError, RuntimeError):
            return False

    sys.path = [path_entry for path_entry in sys.path if not _is_repo_src(path_entry)]

if "load_dotenv" in globals():
    if REPO_ROOT is not None:
        load_dotenv(REPO_ROOT / ".env", override=False)
    else:
        load_dotenv(override=False)

CACHE_ROOT = REPO_ROOT if REPO_ROOT is not None else START
os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(CACHE_ROOT / ".cache" / "perseus-mcp"),
)

for module_name in [
    name
    for name in list(sys.modules)
    if name == "perseus_mcp" or name.startswith("perseus_mcp.")
]:
    del sys.modules[module_name]

from fastmcp import Client
from perseus_mcp import server

server = importlib.reload(server)
mcp = server.mcp

print(f"Install source: {PERSEUS_MCP_INSTALL_SOURCE}")
print(f"Repository root: {REPO_ROOT if REPO_ROOT is not None else 'not found'}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server: {mcp.name}")

## 4 - Result and schema helper functions <a class="anchor" id="helpers"></a>
##### [Back to ToC](#TOC)

These helpers support both the tutorial and generated reference:

- extract text content blocks from a tool result;
- parse tools that return serialized JSON;
- summarize JSON Schema types, including nullable `anyOf` fields;
- distinguish required and optional parameters;
- create compact argument signatures from the live schema.

In [3]:
def tool_text(result):
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


def tool_json(result):
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_json(result)


def schema_type(property_schema):
    """Return a readable type label from a JSON Schema property."""
    if "type" in property_schema:
        return property_schema["type"]
    if "anyOf" in property_schema:
        types = [option.get("type", "complex") for option in property_schema["anyOf"]]
        return " | ".join(types)
    return "complex"


def schema_arguments(tool):
    schema = tool.inputSchema or {}
    required = set(schema.get("required", []))
    rows = []
    for name, details in schema.get("properties", {}).items():
        rows.append(
            {
                "name": name,
                "type": schema_type(details),
                "required": name in required,
                "default": details.get("default", "—"),
                "description": details.get("description", ""),
            }
        )
    return rows


def tool_signature(tool):
    parts = []
    for argument in schema_arguments(tool):
        label = argument["name"]
        if not argument["required"]:
            label = f"[{label}]"
        parts.append(label)
    return f"{tool.name}({', '.join(parts)})"

## 5 - Connect and retrieve the live tool catalog <a class="anchor" id="connect"></a>
##### [Back to ToC](#TOC)

`client.list_tools()` retrieves the same tool definitions exposed to an external MCP client. This call is local: it inspects the registered server surface and does not query Perseus or Scaife.

The result is stored in both list and dictionary form for later reference generation.

In [4]:
async with Client(mcp) as client:
    tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in tools}

print(f"Total registered tools: {len(tools)}")
for index, tool in enumerate(tools, start=1):
    first_line = (tool.description or "No description provided.").splitlines()[0]
    print(f"{index:>2}. {tool_signature(tool)} — {first_line}")

Total registered tools: 23
 1. get_passage(urn) — Get the text of a specific passage using a CTS URN.
 2. get_passage_plus(urn) — Get passage text plus surrounding metadata/context for a CTS URN.
 3. get_passage_plaintext(urn) — Get a passage as plain readable text instead of raw CTS XML.
 4. get_valid_references(urn, [level]) — Get valid citations/references for a work, useful for navigation.
 5. get_valid_references_json(urn, [level], [limit], [offset]) — Get valid citation references as paged JSON instead of raw CTS XML.
 6. count_valid_references(urn, [level]) — Count valid citation references without returning the full reference list.
 7. get_capabilities() — Get the list of available texts and editions from Perseus CTS.
 8. get_cache_status() — Get local metadata cache status.
 9. refresh_metadata_cache() — Refresh cached CTS capabilities metadata from Perseus.
10. clear_metadata_cache() — Clear local metadata cache files and in-memory cache entries.
11. list_text_groups([languag

## 6 - Understand the six tool families <a class="anchor" id="categories"></a>
##### [Back to ToC](#TOC)

The 23 tools are easier to learn when grouped by research purpose rather than alphabetically.

| Family | Core question |
|---|---|
| Passage retrieval | "Give me the text or XML for this CTS passage." |
| References and navigation | "Which citations exist, and what comes before or after this one?" |
| CTS inventory and discovery | "Which authors, works, editions, and translations are available?" |
| Search | "Where does this form, lemma, or expression occur?" |
| Scaife-native retrieval | "Give me Scaife's metadata or passage representation directly." |
| Cache management | "What stable metadata is cached locally, and should it be refreshed or cleared?" |

The category mapping below is maintained as reference metadata. A later drift check ensures every live tool belongs to exactly one category.

In [5]:
TOOL_CATEGORIES = {
    "Passage retrieval": [
        "get_passage",
        "get_passage_plus",
        "get_passage_plaintext",
    ],
    "References and navigation": [
        "get_valid_references",
        "get_valid_references_json",
        "count_valid_references",
        "get_first_urn",
        "get_prev_next_urn",
    ],
    "CTS inventory and discovery": [
        "get_capabilities",
        "list_text_groups",
        "get_author_resources",
        "find_author_names",
        "get_work_resources",
        "get_label",
    ],
    "Search": [
        "search_perseus",
        "search_within_text",
        "get_passage_highlights",
    ],
    "Scaife-native retrieval": [
        "get_scaife_library_metadata",
        "get_scaife_passage_json",
        "get_scaife_passage_text",
    ],
    "Cache management": [
        "get_cache_status",
        "refresh_metadata_cache",
        "clear_metadata_cache",
    ],
}

for category, names in TOOL_CATEGORIES.items():
    print(f"{category} ({len(names)})")
    for name in names:
        print(f"  - {name}")

Passage retrieval (3)
  - get_passage
  - get_passage_plus
  - get_passage_plaintext
References and navigation (5)
  - get_valid_references
  - get_valid_references_json
  - count_valid_references
  - get_first_urn
  - get_prev_next_urn
CTS inventory and discovery (6)
  - get_capabilities
  - list_text_groups
  - get_author_resources
  - find_author_names
  - get_work_resources
  - get_label
Search (3)
  - search_perseus
  - search_within_text
  - get_passage_highlights
Scaife-native retrieval (3)
  - get_scaife_library_metadata
  - get_scaife_passage_json
  - get_scaife_passage_text
Cache management (3)
  - get_cache_status
  - refresh_metadata_cache
  - clear_metadata_cache


## 7 - Learn to read a JSON input schema <a class="anchor" id="schema-reading"></a>
##### [Back to ToC](#TOC)

JSON Schema describes what an argument object may contain. Common fields are:

| Schema field | Meaning |
|---|---|
| `type` | Expected JSON type, such as `string`, `integer`, or `boolean` |
| `properties` | Named arguments accepted by the tool |
| `required` | Arguments that must be supplied |
| `default` | Value used when an optional argument is omitted |
| `anyOf` | Multiple accepted shapes, often a value or `null` |
| `minimum` | Numeric lower bound enforced by validation |
| `description` | Additional guidance for the argument |

The cell below renders `search_perseus`, the most configurable tool, as both raw schema and a simplified parameter table.

In [6]:
search_tool = tool_by_name["search_perseus"]

print("Raw JSON Schema:")
print(json.dumps(search_tool.inputSchema, ensure_ascii=False, indent=2))

print("\nSimplified arguments:")
for argument in schema_arguments(search_tool):
    status = "required" if argument["required"] else "optional"
    print(
        f"- {argument['name']}: {argument['type']} ({status}), "
        f"default={argument['default']!r}"
    )

Raw JSON Schema:
{
  "additionalProperties": false,
  "properties": {
    "query": {
      "type": "string"
    },
    "language": {
      "default": "greek",
      "type": "string"
    },
    "query_format": {
      "default": "auto",
      "type": "string"
    },
    "author": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    },
    "search_kind": {
      "default": "form",
      "type": "string"
    },
    "preserve_operators": {
      "default": false,
      "type": "boolean"
    },
    "page_num": {
      "default": 1,
      "type": "integer"
    },
    "text_group": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    },
    "work": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    },
    "result_

## 8 - Complete quick-reference table <a class="anchor" id="quick-reference"></a>
##### [Back to ToC](#TOC)

The manually maintained metadata below adds information that is not fully encoded in the JSON schema: output representation, upstream source, operational behavior, and a concise usage note.

The subsequent cell combines that metadata with the live tool catalog to generate a complete quick-reference table.

In [7]:
TOOL_NOTES = {
    "get_passage": {"output": "CTS XML", "source": "Perseus CTS", "behavior": "network", "note": "Use when TEI/XML fidelity matters."},
    "get_passage_plus": {"output": "CTS XML", "source": "Perseus CTS", "behavior": "network", "note": "Requests passage plus CTS context/metadata."},
    "get_passage_plaintext": {"output": "plaintext", "source": "Perseus CTS + local parsing", "behavior": "network", "note": "Best default for reading or ordinary Python text processing."},
    "get_valid_references": {"output": "CTS XML", "source": "Perseus CTS", "behavior": "cached network", "note": "Complete raw citation response; can be large."},
    "get_valid_references_json": {"output": "JSON", "source": "Perseus CTS + local paging", "behavior": "cached network", "note": "Preferred paged citation list for agents and interfaces."},
    "count_valid_references": {"output": "JSON", "source": "Perseus CTS + local counting", "behavior": "cached network", "note": "Use when only the number of citations is needed."},
    "get_capabilities": {"output": "CTS XML", "source": "Perseus CTS", "behavior": "cached network", "note": "Complete inventory; large and usually less convenient than discovery helpers."},
    "get_cache_status": {"output": "JSON", "source": "local filesystem/memory", "behavior": "local read", "note": "Safe diagnostic call; no upstream request."},
    "refresh_metadata_cache": {"output": "JSON", "source": "Perseus CTS + local cache", "behavior": "network + local write", "note": "Refreshes cached capabilities metadata."},
    "clear_metadata_cache": {"output": "JSON", "source": "local cache", "behavior": "local delete", "note": "State-changing: removes this server process's memory cache and configured disk cache."},
    "list_text_groups": {"output": "JSON", "source": "Perseus CTS + local filtering", "behavior": "cached network", "note": "Browse authors/textgroups and their works; query can also match work titles."},
    "get_author_resources": {"output": "JSON", "source": "Perseus CTS + local filtering", "behavior": "cached network", "note": "Retrieve works, editions, and translations for an author or textgroup URN."},
    "find_author_names": {"output": "JSON", "source": "Perseus CTS + local filtering", "behavior": "cached network", "note": "Partial-name discovery restricted to author/textgroup name fields."},
    "get_work_resources": {"output": "JSON", "source": "Perseus CTS + local filtering", "behavior": "cached network", "note": "Find editions/translations for a work title or work URN."},
    "get_label": {"output": "upstream XML/HTML", "source": "Perseus CTS", "behavior": "network", "note": "Validate the response; the live service may return HTML instead of label XML."},
    "get_first_urn": {"output": "XML", "source": "Perseus CTS + local fallback", "behavior": "cached network", "note": "Falls back to the first valid reference if upstream navigation is malformed."},
    "get_prev_next_urn": {"output": "XML", "source": "Perseus CTS + local fallback", "behavior": "cached network", "note": "Returns neighboring CTS URNs, deriving them from valid references when needed."},
    "search_perseus": {"output": "Scaife JSON", "source": "Scaife search + optional CTS author resolution", "behavior": "network", "note": "Library-wide form/lemma search with query normalization, pagination, and scopes."},
    "search_within_text": {"output": "Scaife JSON", "source": "Scaife reader search", "behavior": "network", "note": "Search within one selected Scaife edition/text URN."},
    "get_passage_highlights": {"output": "Scaife JSON", "source": "Scaife reader search", "behavior": "network", "note": "Retrieve token-level match positions for one Scaife passage."},
    "get_scaife_library_metadata": {"output": "Scaife JSON", "source": "Scaife library", "behavior": "network", "note": "Metadata for a Scaife textgroup, work, edition, or translation URN."},
    "get_scaife_passage_json": {"output": "Scaife JSON", "source": "Scaife library", "behavior": "network", "note": "Scaife passage metadata/content as JSON."},
    "get_scaife_passage_text": {"output": "plaintext", "source": "Scaife library", "behavior": "network", "note": "Readable Scaife passage text; useful when a Scaife URN has no direct CTS counterpart."},
}

tool_to_category = {
    name: category
    for category, names in TOOL_CATEGORIES.items()
    for name in names
}

quick_rows = []
for tool in tools:
    notes = TOOL_NOTES[tool.name]
    required = [row["name"] for row in schema_arguments(tool) if row["required"]]
    optional = [row["name"] for row in schema_arguments(tool) if not row["required"]]
    quick_rows.append(
        {
            "tool": tool.name,
            "category": tool_to_category[tool.name],
            "required": ", ".join(required) or "—",
            "optional": ", ".join(optional) or "—",
            "output": notes["output"],
            "source": notes["source"],
            "behavior": notes["behavior"],
        }
    )

header = "| Tool | Category | Required | Optional | Output | Source | Behavior |\n|---|---|---|---|---|---|---|"
lines = [header]
for row in quick_rows:
    lines.append(
        f"| `{row['tool']}` | {row['category']} | `{row['required']}` | "
        f"`{row['optional']}` | {row['output']} | {row['source']} | {row['behavior']} |"
    )
display(Markdown("\n".join(lines)))

| Tool | Category | Required | Optional | Output | Source | Behavior |
|---|---|---|---|---|---|---|
| `get_passage` | Passage retrieval | `urn` | `—` | CTS XML | Perseus CTS | network |
| `get_passage_plus` | Passage retrieval | `urn` | `—` | CTS XML | Perseus CTS | network |
| `get_passage_plaintext` | Passage retrieval | `urn` | `—` | plaintext | Perseus CTS + local parsing | network |
| `get_valid_references` | References and navigation | `urn` | `level` | CTS XML | Perseus CTS | cached network |
| `get_valid_references_json` | References and navigation | `urn` | `level, limit, offset` | JSON | Perseus CTS + local paging | cached network |
| `count_valid_references` | References and navigation | `urn` | `level` | JSON | Perseus CTS + local counting | cached network |
| `get_capabilities` | CTS inventory and discovery | `—` | `—` | CTS XML | Perseus CTS | cached network |
| `get_cache_status` | Cache management | `—` | `—` | JSON | local filesystem/memory | local read |
| `refresh_metadata_cache` | Cache management | `—` | `—` | JSON | Perseus CTS + local cache | network + local write |
| `clear_metadata_cache` | Cache management | `—` | `—` | JSON | local cache | local delete |
| `list_text_groups` | CTS inventory and discovery | `—` | `language, query, limit` | JSON | Perseus CTS + local filtering | cached network |
| `get_author_resources` | CTS inventory and discovery | `author` | `language` | JSON | Perseus CTS + local filtering | cached network |
| `find_author_names` | CTS inventory and discovery | `query` | `language, limit` | JSON | Perseus CTS + local filtering | cached network |
| `get_work_resources` | CTS inventory and discovery | `urn_or_title` | `language` | JSON | Perseus CTS + local filtering | cached network |
| `get_label` | CTS inventory and discovery | `urn` | `—` | upstream XML/HTML | Perseus CTS | network |
| `get_first_urn` | References and navigation | `urn` | `—` | XML | Perseus CTS + local fallback | cached network |
| `get_prev_next_urn` | References and navigation | `urn` | `—` | XML | Perseus CTS + local fallback | cached network |
| `search_perseus` | Search | `query` | `language, query_format, author, search_kind, preserve_operators, page_num, text_group, work, result_format` | Scaife JSON | Scaife search + optional CTS author resolution | network |
| `search_within_text` | Search | `query, text_urn` | `language, query_format, search_kind, preserve_operators, size, offset` | Scaife JSON | Scaife reader search | network |
| `get_passage_highlights` | Search | `query, passage_urn` | `language, query_format, search_kind, preserve_operators` | Scaife JSON | Scaife reader search | network |
| `get_scaife_library_metadata` | Scaife-native retrieval | `urn` | `—` | Scaife JSON | Scaife library | network |
| `get_scaife_passage_json` | Scaife-native retrieval | `urn` | `—` | Scaife JSON | Scaife library | network |
| `get_scaife_passage_text` | Scaife-native retrieval | `urn` | `—` | plaintext | Scaife library | network |

## 9 - Choose a tool from a research question <a class="anchor" id="decision-guide"></a>
##### [Back to ToC](#TOC)

| Starting question | Recommended first tool | Why |
|---|---|---|
| "Does this author exist?" | `find_author_names` | Partial-name matching with stable textgroup URNs |
| "What works or editions belong to this author?" | `get_author_resources` | Focused nested resource discovery |
| "What editions exist for this title?" | `get_work_resources` | Work-title/URN lookup |
| "Let me browse authors and works." | `list_text_groups` | Inventory browsing with language/query/limit filters |
| "Give me this passage to read." | `get_passage_plaintext` | Readable text without XML parsing |
| "I need TEI or CTS metadata." | `get_passage` or `get_passage_plus` | Raw source-oriented XML |
| "Does this citation exist?" | `get_valid_references_json` | Paged, structured citation list |
| "How large is the citation inventory?" | `count_valid_references` | Avoids returning the full list |
| "What comes before and after this passage?" | `get_prev_next_urn` | CTS navigation with fallback |
| "Where does this written form occur?" | `search_perseus(search_kind="form")` | Surface-form library search |
| "Where do forms of this headword occur?" | `search_perseus(search_kind="lemma")` | Lemma-index search |
| "Search only this Scaife edition." | `search_within_text` | Reader search scoped to one text URN |
| "Which token matched in this passage?" | `get_passage_highlights` | Passage-level highlight positions |
| "I have a Scaife URN and need its text." | `get_scaife_passage_text` | Avoids assuming it is a valid Perseus CTS URN |
| "Why is discovery slow or stale?" | `get_cache_status` | Inspect before refreshing or clearing |

General rule: **discover before constructing URNs, use structured/paged helpers when available, and keep CTS and Scaife edition URNs in their own service context.**

## 10 - Example argument templates for every tool <a class="anchor" id="examples"></a>
##### [Back to ToC](#TOC)

The following argument objects are examples for `client.call_tool(name, arguments)`. They are not all executed here.

The examples intentionally distinguish:

- CTS edition `perseus-grc1` for CTS-backed passage/navigation calls;
- Scaife edition `perseus-grc2` for Scaife reader, highlight, and passage calls.

Live inventories can change, so production code should discover current URNs rather than copying these examples blindly.

In [8]:
EXAMPLE_ARGUMENTS = {
    "get_passage": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"},
    "get_passage_plus": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"},
    "get_passage_plaintext": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"},
    "get_valid_references": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1", "level": 1},
    "get_valid_references_json": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1", "limit": 10, "offset": 0},
    "count_valid_references": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"},
    "get_capabilities": {},
    "get_cache_status": {},
    "refresh_metadata_cache": {},
    "clear_metadata_cache": {},
    "list_text_groups": {"language": "greek", "query": "Homer", "limit": 10},
    "get_author_resources": {"author": "urn:cts:greekLit:tlg0012", "language": "greek"},
    "find_author_names": {"query": "Hom", "language": "greek", "limit": 10},
    "get_work_resources": {"urn_or_title": "Iliad"},
    "get_label": {"urn": "urn:cts:greekLit:tlg0012.tlg001"},
    "get_first_urn": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"},
    "get_prev_next_urn": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10"},
    "search_perseus": {"query": "μῆνιν", "language": "greek", "query_format": "unicode", "search_kind": "form", "work": "urn:cts:greekLit:tlg0012.tlg001"},
    "search_within_text": {"query": "μῆνιν", "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2", "language": "greek", "query_format": "unicode", "size": 10, "offset": 0},
    "get_passage_highlights": {"query": "μῆνιν", "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1", "language": "greek", "query_format": "unicode"},
    "get_scaife_library_metadata": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2"},
    "get_scaife_passage_json": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1"},
    "get_scaife_passage_text": {"urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1"},
}

for category, names in TOOL_CATEGORIES.items():
    print(f"\n## {category}")
    for name in names:
        print(f"\n{name}")
        print(json.dumps(EXAMPLE_ARGUMENTS[name], ensure_ascii=False, indent=2))


## Passage retrieval

get_passage
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}

get_passage_plus
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}

get_passage_plaintext
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}

## References and navigation

get_valid_references
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": 1
}

get_valid_references_json
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "limit": 10,
  "offset": 0
}

count_valid_references
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"
}

get_first_urn
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"
}

get_prev_next_urn
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10"
}

## CTS inventory and discovery

get_capabilities
{}

list_text_groups
{
  "language": "greek",
  "query": "Homer",
  "limit": 10
}

get_author_resources
{
  "author": "urn:cts:greekLit:tlg0012",
  "language": "greek"
}

find_author_names
{
  "qu

## 11 - Tutorial call: inspect local cache status <a class="anchor" id="local-call"></a>
##### [Back to ToC](#TOC)

`get_cache_status` is a good first tool call because it is local, read-only, and requires no arguments. It demonstrates the full MCP call/result/JSON cycle without depending on an upstream service.

The call result still arrives as content blocks; `call_json` converts the text payload into a Python dictionary.

In [9]:
async with Client(mcp) as client:
    cache_result = await client.call_tool("get_cache_status", {})

print(f"Result object: {type(cache_result).__name__}")
print(f"Content blocks: {len(cache_result.content)}")
print(f"Block types: {[type(block).__name__ for block in cache_result.content]}")
print("\nParsed JSON payload:")
print(json.dumps(tool_json(cache_result), ensure_ascii=False, indent=2))

Result object: CallToolResult
Content blocks: 1
Block types: ['TextContent']

Parsed JSON payload:
{
  "enabled": true,
  "cache_dir": "D:\\Onedrive\\GitHub\\Perseus-mcp\\.cache\\perseus-mcp",
  "ttl_seconds": 86400,
  "memory_entries": 0,
  "disk_files": 4,
  "disk_bytes": 4213423
}


## 12 - Tutorial call: discover an author <a class="anchor" id="live-call"></a>
##### [Back to ToC](#TOC)

`find_author_names` demonstrates a cached upstream workflow. The first call may fetch the large CTS capabilities inventory; later discovery calls can reuse the configured metadata cache.

The compact display keeps stable identifiers and removes the nested work details from the first view. The complete result remains available in `author_results`.

In [10]:
async with Client(mcp) as client:
    author_results = await call_json(
        client,
        "find_author_names",
        EXAMPLE_ARGUMENTS["find_author_names"],
    )

compact_authors = [
    {
        "urn": author["urn"],
        "names": author["names"],
        "matched_names": author["matched_names"],
        "works_count": author["works_count"],
    }
    for author in author_results["authors"]
]
print(json.dumps(compact_authors, ensure_ascii=False, indent=2))

[
  {
    "urn": "urn:cts:greekLit:tlg0013",
    "names": [
      "Homeric Hymns"
    ],
    "matched_names": [
      "Homeric Hymns"
    ],
    "works_count": 33
  },
  {
    "urn": "urn:cts:greekLit:tlg0012",
    "names": [
      "Homer"
    ],
    "matched_names": [
      "Homer"
    ],
    "works_count": 2
  }
]


## 13 - Parse JSON, XML, plaintext, and upstream responses <a class="anchor" id="result-types"></a>
##### [Back to ToC](#TOC)

| Representation | Typical tools | Caller action |
|---|---|---|
| Serialized JSON | Discovery, search, Scaife JSON, cache, paged references | `json.loads(tool_text(result))` |
| XML | Raw CTS passages, capabilities, references, navigation | Parse with `xml.etree.ElementTree` or another XML library |
| Plaintext | `get_passage_plaintext`, `get_scaife_passage_text` | Use directly as a string |
| Unexpected upstream HTML | Currently possible for routes such as `GetLabel` | Validate format before XML parsing |

Tool success at the MCP transport level means the tool returned content; it does not always prove that an imperfect upstream service returned the ideal semantic format. Defensive callers should inspect or parse the expected representation.

For JSON-returning tools, avoid truncating the string before parsing. Parse first, then select the fields needed for display.

## 14 - Validate arguments before calling <a class="anchor" id="validation"></a>
##### [Back to ToC](#TOC)

FastMCP validates calls against the tool schema, but client code can catch obvious omissions earlier. The helper below checks required keys and unknown keys using the live schema.

This is only a lightweight preflight check. It does not replace complete JSON Schema validation, type checking, enum checking, numeric bounds, or server-side semantic validation.

In [11]:
def preflight_arguments(tool, arguments):
    schema = tool.inputSchema or {}
    properties = set(schema.get("properties", {}))
    required = set(schema.get("required", []))
    supplied = set(arguments)
    return {
        "tool": tool.name,
        "missing_required": sorted(required - supplied),
        "unknown_arguments": sorted(supplied - properties),
        "ready_for_full_validation": not (required - supplied) and not (supplied - properties),
    }


print("Valid example:")
print(
    json.dumps(
        preflight_arguments(
            tool_by_name["get_passage_plaintext"],
            EXAMPLE_ARGUMENTS["get_passage_plaintext"],
        ),
        indent=2,
    )
)

print("\nInvalid example, not sent to the server:")
print(
    json.dumps(
        preflight_arguments(
            tool_by_name["get_passage_plaintext"],
            {"passage": "1.1"},
        ),
        indent=2,
    )
)

Valid example:
{
  "tool": "get_passage_plaintext",
  "missing_required": [],
  "unknown_arguments": [],
  "ready_for_full_validation": true
}

Invalid example, not sent to the server:
{
  "tool": "get_passage_plaintext",
  "missing_required": [
    "urn"
  ],
  "unknown_arguments": [
    "passage"
  ],
  "ready_for_full_validation": false
}


## 15 - Search and filter the catalog programmatically <a class="anchor" id="catalog-query"></a>
##### [Back to ToC](#TOC)

Once the catalog is in Python, it can answer questions about the tool surface itself. The helper searches names, descriptions, category labels, output formats, sources, behavior, and argument names.

This is useful for documentation interfaces, command palettes, agent planning, and regression tests.

In [12]:
def search_tool_catalog(query):
    needle = query.casefold()
    matches = []
    for tool in tools:
        notes = TOOL_NOTES[tool.name]
        arguments = schema_arguments(tool)
        searchable = " ".join(
            [
                tool.name,
                tool.description or "",
                tool_to_category[tool.name],
                notes["output"],
                notes["source"],
                notes["behavior"],
                notes["note"],
                *[argument["name"] for argument in arguments],
            ]
        ).casefold()
        if needle in searchable:
            matches.append(
                {
                    "tool": tool.name,
                    "category": tool_to_category[tool.name],
                    "signature": tool_signature(tool),
                    "output": notes["output"],
                    "behavior": notes["behavior"],
                }
            )
    return matches


for query in ["lemma", "cache", "plaintext", "offset"]:
    print(f"\nCatalog query: {query!r}")
    print(json.dumps(search_tool_catalog(query), ensure_ascii=False, indent=2))


Catalog query: 'lemma'
[
  {
    "tool": "search_perseus",
    "category": "Search",
    "signature": "search_perseus(query, [language], [query_format], [author], [search_kind], [preserve_operators], [page_num], [text_group], [work], [result_format])",
    "output": "Scaife JSON",
    "behavior": "network"
  }
]

Catalog query: 'cache'
[
  {
    "tool": "get_valid_references",
    "category": "References and navigation",
    "signature": "get_valid_references(urn, [level])",
    "output": "CTS XML",
    "behavior": "cached network"
  },
  {
    "tool": "get_valid_references_json",
    "category": "References and navigation",
    "signature": "get_valid_references_json(urn, [level], [limit], [offset])",
    "output": "JSON",
    "behavior": "cached network"
  },
  {
    "tool": "count_valid_references",
    "category": "References and navigation",
    "signature": "count_valid_references(urn, [level])",
    "output": "JSON",
    "behavior": "cached network"
  },
  {
    "tool": "get_ca

## 16 - Detect documentation and server drift <a class="anchor" id="drift-check"></a>
##### [Back to ToC](#TOC)

A reference notebook can become stale when tools are added, removed, or renamed. The checks below compare the live catalog with:

- the category mapping;
- the operational notes;
- the example argument templates.

The assertions intentionally fail if a future tool change is not reflected in this handbook.

In [13]:
live_names = set(tool_by_name)
category_names = {
    name for names in TOOL_CATEGORIES.values() for name in names
}
note_names = set(TOOL_NOTES)
example_names = set(EXAMPLE_ARGUMENTS)
example_schema_issues = {}
for name in live_names & example_names:
    report = preflight_arguments(
        tool_by_name[name],
        EXAMPLE_ARGUMENTS[name],
    )
    if not report["ready_for_full_validation"]:
        example_schema_issues[name] = report

category_duplicates = [
    name
    for name, count in Counter(
        name for names in TOOL_CATEGORIES.values() for name in names
    ).items()
    if count > 1
]

drift_report = {
    "live_tool_count": len(live_names),
    "missing_from_categories": sorted(live_names - category_names),
    "unknown_in_categories": sorted(category_names - live_names),
    "category_duplicates": category_duplicates,
    "missing_notes": sorted(live_names - note_names),
    "unknown_notes": sorted(note_names - live_names),
    "missing_examples": sorted(live_names - example_names),
    "unknown_examples": sorted(example_names - live_names),
    "example_schema_issues": example_schema_issues,
}
print(json.dumps(drift_report, indent=2))

assert not any(
    value
    for key, value in drift_report.items()
    if key != "live_tool_count"
), "The live MCP surface and notebook reference metadata have drifted apart."

{
  "live_tool_count": 23,
  "missing_from_categories": [],
  "unknown_in_categories": [],
  "category_duplicates": [],
  "missing_notes": [],
  "unknown_notes": [],
  "missing_examples": [],
  "unknown_examples": [],
  "example_schema_issues": {}
}


## 17 - Detailed generated reference for all tools <a class="anchor" id="detailed-reference"></a>
##### [Back to ToC](#TOC)

The next cell generates the long-form reference from three sources:

1. live FastMCP descriptions and schemas;
2. the maintained category/output/behavior notes;
3. the example argument templates.

Each tool card includes its signature, description, operational notes, parameter table, raw input schema, and example call arguments. This is intentionally comprehensive.

In [14]:
for category, names in TOOL_CATEGORIES.items():
    display(Markdown(f"### {category}"))
    for name in names:
        tool = tool_by_name[name]
        notes = TOOL_NOTES[name]
        arguments = schema_arguments(tool)

        if arguments:
            parameter_lines = [
                "| Argument | Type | Status | Default | Description |",
                "|---|---|---|---|---|",
            ]
            for argument in arguments:
                status = "required" if argument["required"] else "optional"
                description = argument["description"].replace("|", "\\|")
                parameter_lines.append(
                    f"| `{argument['name']}` | `{argument['type']}` | {status} | "
                    f"`{argument['default']}` | {description} |"
                )
            parameter_table = "\n".join(parameter_lines)
        else:
            parameter_table = "*No arguments.*"

        schema_json = json.dumps(tool.inputSchema, ensure_ascii=False, indent=2)
        example_json = json.dumps(EXAMPLE_ARGUMENTS[name], ensure_ascii=False, indent=2)
        description = tool.description or "No description provided."

        display(
            Markdown(
                f"#### `{name}`\n\n"
                f"**Signature:** `{tool_signature(tool)}`  \n"
                f"**Output:** {notes['output']}  \n"
                f"**Source:** {notes['source']}  \n"
                f"**Behavior:** {notes['behavior']}  \n"
                f"**Reference note:** {notes['note']}\n\n"
                f"{description}\n\n"
                f"**Parameters**\n\n{parameter_table}\n\n"
                f"**Example arguments**\n\n```json\n{example_json}\n```\n\n"
                f"<details><summary>Raw input schema</summary>\n\n"
                f"```json\n{schema_json}\n```\n\n</details>"
            )
        )

### Passage retrieval

#### `get_passage`

**Signature:** `get_passage(urn)`  
**Output:** CTS XML  
**Source:** Perseus CTS  
**Behavior:** network  
**Reference note:** Use when TEI/XML fidelity matters.

Get the text of a specific passage using a CTS URN.

Examples:
- urn:cts:greekLit:tlg0012.tlg001:1.1-1.10
- urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1

**Parameters**

| Argument | Type | Status | Default | Description |
|---|---|---|---|---|
| `urn` | `string` | required | `—` |  |

**Example arguments**

```json
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

</details>

#### `get_passage_plus`

**Signature:** `get_passage_plus(urn)`  
**Output:** CTS XML  
**Source:** Perseus CTS  
**Behavior:** network  
**Reference note:** Requests passage plus CTS context/metadata.

Get passage text plus surrounding metadata/context for a CTS URN.

**Parameters**

| Argument | Type | Status | Default | Description |
|---|---|---|---|---|
| `urn` | `string` | required | `—` |  |

**Example arguments**

```json
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

</details>

#### `get_passage_plaintext`

**Signature:** `get_passage_plaintext(urn)`  
**Output:** plaintext  
**Source:** Perseus CTS + local parsing  
**Behavior:** network  
**Reference note:** Best default for reading or ordinary Python text processing.

Get a passage as plain readable text instead of raw CTS XML.

**Parameters**

| Argument | Type | Status | Default | Description |
|---|---|---|---|---|
| `urn` | `string` | required | `—` |  |

**Example arguments**

```json
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

</details>

### References and navigation

...

#### `get_scaife_passage_text`

**Signature:** `get_scaife_passage_text(urn)`  
**Output:** plaintext  
**Source:** Scaife library  
**Behavior:** network  
**Reference note:** Readable Scaife passage text; useful when a Scaife URN has no direct CTS counterpart.

Get Scaife plaintext for a passage URN.

**Parameters**

| Argument | Type | Status | Default | Description |
|---|---|---|---|---|
| `urn` | `string` | required | `—` |  |

**Example arguments**

```json
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1"
}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {
    "urn": {
      "type": "string"
    }
  },
  "required": [
    "urn"
  ],
  "type": "object"
}
```

</details>

### Cache management

#### `get_cache_status`

**Signature:** `get_cache_status()`  
**Output:** JSON  
**Source:** local filesystem/memory  
**Behavior:** local read  
**Reference note:** Safe diagnostic call; no upstream request.

Get local metadata cache status.

**Parameters**

*No arguments.*

**Example arguments**

```json
{}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {},
  "type": "object"
}
```

</details>

#### `refresh_metadata_cache`

**Signature:** `refresh_metadata_cache()`  
**Output:** JSON  
**Source:** Perseus CTS + local cache  
**Behavior:** network + local write  
**Reference note:** Refreshes cached capabilities metadata.

Refresh cached CTS capabilities metadata from Perseus.

**Parameters**

*No arguments.*

**Example arguments**

```json
{}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {},
  "type": "object"
}
```

</details>

#### `clear_metadata_cache`

**Signature:** `clear_metadata_cache()`  
**Output:** JSON  
**Source:** local cache  
**Behavior:** local delete  
**Reference note:** State-changing: removes this server process's memory cache and configured disk cache.

Clear local metadata cache files and in-memory cache entries.

**Parameters**

*No arguments.*

**Example arguments**

```json
{}
```

<details><summary>Raw input schema</summary>

```json
{
  "additionalProperties": false,
  "properties": {},
  "type": "object"
}
```

</details>

## 18 - Build a machine-readable reference catalog <a class="anchor" id="machine-readable"></a>
##### [Back to ToC](#TOC)

Applications often need structured metadata rather than rendered Markdown. The cell combines FastMCP's model dump with the handbook metadata and example arguments.

The resulting `reference_catalog` can be filtered, serialized, used in tests, or adapted for a documentation generator. It is printed only in abbreviated form to avoid duplicating the full detailed reference in notebook output.

In [15]:
reference_catalog = []
for tool in tools:
    reference_catalog.append(
        {
            **tool.model_dump(),
            "category": tool_to_category[tool.name],
            "reference": TOOL_NOTES[tool.name],
            "example_arguments": EXAMPLE_ARGUMENTS[tool.name],
            "simplified_arguments": schema_arguments(tool),
        }
    )

print(f"Reference entries: {len(reference_catalog)}")
print("Fields in one entry:")
print(sorted(reference_catalog[0]))
print("\nFirst abbreviated entry:")
print(
    json.dumps(
        {
            "name": reference_catalog[0]["name"],
            "category": reference_catalog[0]["category"],
            "reference": reference_catalog[0]["reference"],
            "example_arguments": reference_catalog[0]["example_arguments"],
        },
        ensure_ascii=False,
        indent=2,
    )
)

Reference entries: 23
Fields in one entry:
['annotations', 'category', 'description', 'example_arguments', 'execution', 'icons', 'inputSchema', 'meta', 'name', 'outputSchema', 'reference', 'simplified_arguments', 'title']

First abbreviated entry:
{
  "name": "get_passage",
  "category": "Passage retrieval",
  "reference": {
    "output": "CTS XML",
    "source": "Perseus CTS",
    "behavior": "network",
    "note": "Use when TEI/XML fidelity matters."
  },
  "example_arguments": {
    "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1"
  }
}


## 19 - Network, cache, cost, and safety considerations <a class="anchor" id="operations"></a>
##### [Back to ToC](#TOC)

The MCP server itself does not charge money and requires no API key. However, tool calls have different operational characteristics:

- **Local reads:** `get_cache_status` is fast and does not use the network.
- **Cached metadata:** capabilities and valid-reference tools may make a large first request, then reuse memory/disk cache entries.
- **Ordinary network calls:** passage, search, Scaife retrieval, and label calls contact live public services.
- **Local writes:** cached metadata calls can create files under `PERSEUS_MCP_CACHE_DIR`.
- **State-changing local operation:** `clear_metadata_cache` removes configured cache files and in-memory entries. Do not call it merely to demonstrate syntax.
- **Potentially large payloads:** raw capabilities and complete valid-reference XML can be multi-megabyte responses. Prefer discovery, paging, and count helpers.
- **No automatic batching guarantee:** repeated passage/search calls still create repeated upstream requests; design large workflows responsibly.

Live upstream data can change. For reproducible research, record tool name, arguments, relevant URNs, execution date, and the returned evidence.

## 20 - Use the same catalog from external MCP clients <a class="anchor" id="external-clients"></a>
##### [Back to ToC](#TOC)

This notebook uses an in-process transport, but external MCP hosts see the same tool names, descriptions, and schemas when they launch the `perseus-mcp` command over stdio.

Typical launch command:

```bash
uv --directory /full/path/to/Perseus-mcp run perseus-mcp
```

An external client can use the catalog to:

- display a tool picker;
- let a model select a tool;
- construct a JSON argument object;
- validate required fields;
- send the call over MCP;
- return content blocks to the user or model.

The notebook does not require or configure a particular desktop client. See [`docs/enduser.md`](../docs/enduser.md) and the project README for host-specific setup guidance.

## 21 - Troubleshooting <a class="anchor" id="troubleshooting"></a>
##### [Back to ToC](#TOC)

| Symptom | Response |
|---|---|
| `fastmcp` cannot be imported | Run the installation cell in the active kernel or install the project |
| The catalog shows an old tool count | Rerun setup to reload `perseus_mcp.server`; restart the kernel if needed |
| Drift-check assertion fails | Add/remove the tool in categories, notes, and examples so the handbook matches the server |
| A required argument is unclear | Inspect the simplified parameter table and raw schema |
| A tool returns text that will not parse as JSON | Confirm its expected output format; some tools return XML or plaintext |
| XML parsing fails on an upstream response | Inspect the text prefix; routes such as `GetLabel` may return HTML |
| Search results use an unfamiliar edition URN | Determine whether it is a Scaife edition and do not pass it blindly to CTS tools |
| Discovery is slow on the first run | Large CTS metadata may be downloading; inspect cache status afterward |
| A cache file causes read errors | Use a local writable cache path, or clear the configured metadata cache deliberately |
| An external client has no tools | Verify its command, repository directory, Python environment, and restart the host |

When debugging a call, capture the tool name, argument object, expected representation, exception text, and whether the failure occurred before or after contacting an upstream service.

## 22 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Use this notebook as a reference while working through focused examples:

- [`03_mcp_connection_homer_iliad.ipynb`](03_mcp_connection_homer_iliad.ipynb) — discovery, references, and passage retrieval;
- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) — Unicode/Beta Code search, result interpretation, URN mapping, and navigation;
- [`07_mcp_advanced_search_options.ipynb`](07_mcp_advanced_search_options.ipynb) — form/lemma comparison and operator-preserving search;
- [`08_mcp_cache_and_search_tools.ipynb`](08_mcp_cache_and_search_tools.ipynb) — cache controls, paging, reader search, highlights, and Scaife-native retrieval;
- [`06_openrouter_llm_mcp_interaction.ipynb`](06_openrouter_llm_mcp_interaction.ipynb) — convert the same tool catalog into an LLM tool-calling loop.

When a new tool is added to `perseus_mcp.server`, update the category mapping, operational notes, and example arguments here. The drift check will identify any omission.

## 23 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This handbook is generated from and describes:

- the live local MCP tool registry loaded from [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
- project behavior documented in the [README](../README.md);
- [FastMCP](https://github.com/jlowin/fastmcp), which provides the client, server, tool registry, schemas, validation, and transports;
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) CTS service used by CTS-backed tools;
- the [Scaife Viewer](https://scaife.perseus.org/) services used by search and Scaife-native retrieval tools.

Descriptions and schemas are taken from the running server. Categories, output types, operational notes, and examples are maintained in this notebook and checked against the live tool names.

## 24 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. Runtime dependencies are:

- `fastmcp>=2.12.0`;
- `httpx>=0.27.0`;
- `python-dotenv>=1.0.0`;
- Jupyter/IPython for notebook execution and rendered reference cards.

`collections`, `importlib`, `json`, `os`, `pathlib`, and `sys` are Python standard-library modules.

Recommended installation:

```bash
pip install -e .
```

or:

```bash
uv sync
```

## 25 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.3</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>